In [54]:
# --- apiModule.py ---
import pandas as pd
import numpy as np
from pycaret.regression import load_model, predict_model
from joblib import load

# Load datasets
forecast_df = pd.read_csv("scoring_system/datasets/synthetic_features_with_noise_2026_2050.csv")
static_df = pd.read_csv("scoring_system/datasets/static/static_features_uganda_cities_.csv")

# Load models
models = {
    "MonsoonIntensity": load_model("scoring_system/best_model_monsoon_intensity"),
    "ClimateChange": load_model("scoring_system/best_model_climate_change"),
    "Siltation": load_model("scoring_system/best_model_siltation"),
    "AgriculturalPractices": load_model("scoring_system/best_model_agricultural_practices"),
    "Landslides": load_model("scoring_system/best_model_landslide_risks")
}

def get_prediction_dataframe(city, date):
    city = city.capitalize()  # Standardize casing
    forecast_row = forecast_df[(forecast_df['city'] == city) & (forecast_df['date'] == date)]
    if forecast_row.empty:
        raise ValueError("No forecast data found for given city and date.")

    # Predict using models (rounded int output)
    predictions = {}
    for name, model in models.items():
        pred = predict_model(model, data=forecast_row)
        predictions[name] = int(round(pred.iloc[0, -1]))

    static_row = static_df[static_df['City'] == city]
    if static_row.empty:
        raise ValueError("No static data found for given city.")

    static_features = static_row.drop(columns=['City']).iloc[0].to_dict()

    final_data = {
        "MonsoonIntensity": predictions.get("MonsoonIntensity"),
        "ClimateChange": predictions.get("ClimateChange"),
        "Siltation": predictions.get("Siltation"),
        "AgriculturalPractices": predictions.get("AgriculturalPractices"),
        "Landslides": predictions.get("Landslides"),
        **static_features
    }

    final_columns = [
        "MonsoonIntensity", "TopographyDrainage", "RiverManagement", "Deforestation", "Urbanization",
        "ClimateChange", "DamsQuality", "Siltation", "AgriculturalPractices", "Encroachments",
        "IneffectiveDisasterPreparedness", "DrainageSystems", "CoastalVulnerability", "Landslides",
        "Watersheds", "DeterioratingInfrastructure", "PopulationScore", "WetlandLoss",
        "InadequatePlanning", "PoliticalFactors"
    ]

    final_df = pd.DataFrame([{col: final_data.get(col, None) for col in final_columns}])
    scaler = load('Core_system/scaler.pkl')
    res_df = scaler.transform(final_df)
    res_df = pd.DataFrame(res_df, columns=final_columns)

    # Feature Engineering
    res_df['RunoffPotential'] = (
        res_df['MonsoonIntensity'] + res_df['Urbanization'] + res_df['Deforestation'] +
        res_df['AgriculturalPractices'] + res_df['Siltation']
    ) / 5

    res_df['DrainageCapacity'] = (
        res_df['TopographyDrainage'] + res_df['RiverManagement'] +
        res_df['DrainageSystems'] + res_df['DamsQuality']
    ) / 4

    res_df['FloodSpreadPotential'] = (
        res_df['WetlandLoss'] + res_df['Encroachments'] +
        res_df['CoastalVulnerability'] + (1 - res_df['Watersheds'])
    ) / 4

    res_df['VulnerabilityIndex'] = (
        res_df['PopulationScore'] + res_df['InadequatePlanning'] +
        res_df['IneffectiveDisasterPreparedness'] + res_df['PoliticalFactors']
    ) / 4

    res_df['FloodSizeScore'] = (
        res_df['RunoffPotential'] + res_df['FloodSpreadPotential'] - res_df['DrainageCapacity']
    )

    flood_model = load('Core_system/flood_prediction_model.pkl')
    flood_prediction = flood_model.predict(res_df)

    print("Prediction type:", type(flood_prediction))
    print("Prediction content sample:", flood_prediction)
    print("Prediction shape:", flood_prediction.shape if isinstance(flood_prediction, np.ndarray) else "Not a NumPy array")

    FloodProbability, FloodSizeScore, VulnerabilityIndex = flood_prediction[0]

    result_df = pd.DataFrame([{
        "FloodProbability": FloodProbability,
        "FloodSizeScore": FloodSizeScore,
        "VulnerabilityIndex": VulnerabilityIndex
    }])


    return pd.concat([res_df, result_df], axis=1)

# Example usage:
df = get_prediction_dataframe("Kampala", "2026-01-01")
print(df.head())

Transformation Pipeline and Model Successfully Loaded
Transformation Pipeline and Model Successfully Loaded
Transformation Pipeline and Model Successfully Loaded
Transformation Pipeline and Model Successfully Loaded
Transformation Pipeline and Model Successfully Loaded
Prediction type: <class 'numpy.ndarray'>
Prediction content sample: [[0.915      0.58117944 0.75740132]]
Prediction shape: (1, 3)
   MonsoonIntensity  TopographyDrainage  RiverManagement  Deforestation  \
0              0.25            0.388889              0.5       0.705882   

   Urbanization  ClimateChange  DamsQuality  Siltation  AgriculturalPractices  \
0      0.882353       0.235294        0.375        0.0                  0.625   

   Encroachments  ...  InadequatePlanning  PoliticalFactors  RunoffPotential  \
0       0.777778  ...               0.875             0.625         0.492647   

   DrainageCapacity  FloodSpreadPotential  VulnerabilityIndex  FloodSizeScore  \
0          0.448325              0.536858   